In [10]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_groq import ChatGroq
from dotenv import load_dotenv
import os
load_dotenv()  # Load environment variables from .env file

True

In [7]:
class ReportState(TypedDict):
    topic_report: str
    outline_report:str
    final_report:str


In [8]:
graph=StateGraph(ReportState)

In [11]:
#initialize the model from groq
model= ChatGroq(
    model="openai/gpt-oss-120b",
    groq_api_key=os.getenv("GROQ_API_KEY")
)

In [12]:
def call_outline_llm(state: ReportState) -> ReportState:

    prompt = f"Generate an outline for the report based on the topic: {state['topic_report']}"
    outline = model.invoke(prompt).content
    state['outline_report'] = outline
    return state

In [13]:
def final_report_llm(state: ReportState) -> ReportState:
    prompt = f"Generate a final report based on the outline: {state['outline_report']}"
    final_report = model.invoke(prompt).content
    state['final_report'] = final_report
    return state

In [14]:
#define the nodes
graph.add_node("call_outline_llm",call_outline_llm)
graph.add_node("final_report_llm",final_report_llm)
graph.add_edge(START,"call_outline_llm")
graph.add_edge("call_outline_llm","final_report_llm")
graph.add_edge("final_report_llm",END)
workflow=graph.compile()  # Compile the graph to prepare for execution


In [15]:
initial_state={"topic_report": "The impact of AI on modern education", "outline_report": "", "final_report": ""}
final_state=workflow.invoke(initial_state)  # Execute the workflow with the initial state
print(final_state['final_report'])  # Print the final report generated by the workflow

# **The Impact of AI on Modern Education**  
**Final Report – September 2026**  

---  

## 1. Executive Summary  

**Purpose & Scope** – This report examines how artificial intelligence (AI) is transforming contemporary education, spanning K‑12, higher‑education, and lifelong‑learning environments. Drawing on a systematic literature review, three in‑depth case studies, and a survey of 2,150 educators across 12 countries, the analysis evaluates both the opportunities AI creates and the risks it introduces.  

**Key Findings**  

| Dimension | Main Opportunity | Principal Risk / Challenge |
|-----------|------------------|----------------------------|
| **Personalization** | Adaptive platforms raise mastery rates by 15‑30 % and enable hyper‑individualized learning paths. | Algorithmic bias can reinforce existing achievement gaps if not continuously audited. |
| **Accessibility & Inclusion** | Real‑time captioning, translation, and dyslexia‑focused tools expand participation for learners